In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json" # file unico (30 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=5, top_k=10, top_n=7,       # stesso numero di chunk (top_j) per TUTTE le configurazioni
    # gold_path=QDIR / "gold_chunks.json",   # opzionale: {query_id: [chunk_id,...]} -> metriche retrieval senza LLM
)
print("!>> Handler e benchmark pronti")

<<! File sidecar ripristinato allo stato iniziale !>>


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
08:03:27 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
08:03:27 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> Handler e benchmark pronti


In [2]:
### 1 ### Unione dei due file in eval_queries.json (serve a handler.update_query(new_id=...)) + controlli
import json
from pathlib import Path
QDIR = Path("~/tesi_graphrag/data/queries/dst").expanduser()

pred = json.load(open(QDIR / "queries_predetermined.json", encoding="utf-8"))
norm = json.load(open(QDIR / "queries_normal.json", encoding="utf-8"))
allq = pred + norm

ids = [q["id"] for q in allq]
assert len(ids) == len(set(ids)) == 30, "id duplicati o numero query inatteso"
assert all(q["query"].strip() and q["reference"].strip() for q in allq)

json.dump(allq, open(QDIR / "eval_queries.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
q_by_id = {q["id"]: q for q in allq}                 # accesso per chiave, es. q_by_id["SPE_P03"]["reference"]
print(len(pred), "predeterminate +", len(norm), "normali ->", len(allq), "in eval_queries.json")

15 predeterminate + 15 normali -> 30 in eval_queries.json


In [3]:
### 2 ### Registrazione delle predeterminate nel sidecar (ids: predq_AMB_P01, predq_SPE_P03, ...)
registered = bench.register_predetermined()
print(len(registered), "predeterminate registrate (eseguire UNA sola volta per sidecar)")

08:03:28 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_AMB_P01] |>>
08:03:28 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_AMB_P02] |>>
08:03:29 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_AMB_P03] |>>
08:03:29 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_AMB_P04] |>>
08:03:29 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_AMB_P05] |>>
08:03:30 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_SPE_P01] |>>
08:03:30 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_SPE_P02] |>>
08:03:31 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_SPE_P03] |>>
08:03:31 [INFO] src.predetermined_query_handling

15 predeterminate registrate (eseguire UNA sola volta per sidecar)


In [4]:
### 3 ### Controllo similarita': nessuna normale deve avvicinarsi a una predeterminata (soglia handler 0.85)
sim = bench.check_similarity(warn_above=0.80)
display(sim.head(10))       # le piu' vicine; quelle >= 0.80 vanno riformulate

,normal_id,paired_with,max_sim,closest_pred,sim_with_partner
8,SPE_N04,SPE_P04,0.777523,SPE_P03,0.737325
10,SPE_N06,SPE_P06,0.766059,SPE_P06,0.766059
13,SPE_N09,SPE_P09,0.736811,SPE_P09,0.736811
5,SPE_N01,SPE_P01,0.673757,SPE_P01,0.673757
12,SPE_N08,SPE_P08,0.670615,SPE_P07,0.565023
1,AMB_N02,AMB_P02,0.623668,AMB_P02,0.623668
14,SPE_N10,SPE_P10,0.621166,SPE_P10,0.621166
11,SPE_N07,SPE_P07,0.602452,SPE_P07,0.602452
2,AMB_N03,AMB_P03,0.555792,SPE_P10,0.156779
9,SPE_N05,SPE_P05,0.547597,AMB_P05,0.340586
